# Cryptography Lab Analysis

This notebook analyzes the benchmark data generated by `src/generate_data.py`.
It covers classical vs post-quantum algorithm performance, key/signature sizes,
and NIST security level mapping.

**Run first:** `python src/generate_data.py` to populate `data/`.

In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path('../data')
print('Data directory:', DATA.resolve())
print('Files:', list(DATA.glob('*.csv')))

## Algorithm Benchmark

Load and inspect the crypto benchmark dataset.

In [ ]:
bench_path = DATA / 'crypto_benchmark.csv'
if bench_path.exists():
    df = pd.read_csv(bench_path)
    df['quantum_safe'] = df['quantum_safe'].map({'True': True, 'False': False})
    print(f'Loaded {len(df)} rows')
    print(df.head(10).to_string())
else:
    print('Run generate_data.py first')
    df = None

## Security Level Comparison

NIST Post-Quantum Security Levels (FIPS 203/204/205):

| Level | Equivalent Classical | Description |
|-------|---------------------|-------------|
| 1     | AES-128             | 128-bit quantum security |
| 2     | SHA-256 preimage    | 128-bit collision security |
| 3     | AES-192             | 192-bit quantum security |
| 4     | SHA-384 preimage    | 192-bit collision security |
| 5     | AES-256             | 256-bit quantum security |

### Recommended Algorithms per NIST (2024)

| Algorithm     | Standard  | Level | Use Case        | Key (bytes) | Sig/CT (bytes) |
|---------------|-----------|-------|-----------------|-------------|----------------|
| ML-KEM-768    | FIPS 203  | 3     | Key encapsulation | 1184      | 1088           |
| ML-DSA-65     | FIPS 204  | 3     | Digital signatures | 1952     | 3293           |
| SLH-DSA-128f  | FIPS 205  | 1     | Stateless hash sig | 32       | 49856          |
| FALCON-512    | TBD       | 1     | Compact lattice sig | 897     | 666            |

In [ ]:
if df is not None:
    # Show quantum-safe vs vulnerable summary
    summary = df.groupby('quantum_safe').agg(
        count=('algorithm', 'count'),
        avg_keygen_ms=('keygen_ms', 'mean'),
        avg_sign_ms=('sign_ms', 'mean'),
    ).round(4)
    print('\nSummary by quantum_safe flag:')
    print(summary.to_string())
    
    # Vulnerability scan
    vscan_path = DATA / 'vulnerability_scan.csv'
    if vscan_path.exists():
        vs = pd.read_csv(vscan_path)
        print('\nVulnerability scan priority breakdown:')
        print(vs['priority'].value_counts().to_string())
else:
    print('No data loaded — run generate_data.py')

## Key and Signature Size Overhead

Comparing wire overhead of PQC algorithms vs classical equivalents.

| Metric                    | ECDSA-P256 | ML-DSA-65  | Overhead Factor |
|---------------------------|------------|------------|-----------------|
| Public key bytes          | 64         | 1952       | 30.5×           |
| Signature bytes           | 64         | 3293       | 51.5×           |
| Key generation (ms)       | 0.30       | 0.14       | 0.47× (faster!) |
| Sign (ms)                 | 0.20       | 0.35       | 1.75×           |
| Verify (ms)               | 0.50       | 0.14       | 0.28× (faster!) |

Key insight: ML-DSA is actually **faster** than ECDSA for keygen and verify.
The main cost is the larger key/signature sizes increasing bandwidth.